In [3]:
import pandas as pd     


In [4]:
componentes = pd.read_parquet(r'C:\Users\Alef\Documents\Aprendizado de Máquina\dados\componentes.parquet')
cursos = pd.read_parquet(r'C:\Users\Alef\Documents\Aprendizado de Máquina\dados\cursos.parquet')
discentes = pd.read_parquet(r'C:\Users\Alef\Documents\Aprendizado de Máquina\dados\discentes.parquet')
matriculas = pd.read_parquet(r'C:\Users\Alef\Documents\Aprendizado de Máquina\dados\matriculas.parquet')

# não existe relacionamento por id, solução: criar um agrupametno por em nivel em departamento
docentes = pd.read_parquet(r'C:\Users\Alef\Documents\Aprendizado de Máquina\dados\docentes.parquet')

In [5]:
# garantir unicidade das linhas
discentes.drop_duplicates(subset=['id_discente'], inplace=True, ignore_index=True)

df = (
    discentes.merge(matriculas, on ='id_discente', how='right')
    .merge(componentes, on='id_disciplina',how= 'left')
    .merge(cursos, on = ['id_estrutura_curricular','id_disciplina'], how='left', suffixes=['','_cursos'])
)

In [6]:
df1 = df.copy()

col = list(discentes.columns) + ['situacao', 'id_disciplina', 'id_curso_cursos','ano','periodo','curso_nome']
x1 =(df1[col])

In [7]:
df1 = df.copy()

col = list(discentes.columns) + ['situacao', 'id_disciplina', 'id_curso_cursos','ano','periodo','ch_total','nome_componete_curricular','curso_nome']
x1 =(df1[col])

# funções auxiliares

def gerar_dummies(df,coluna, map=None):
    if map is not None:
        df[coluna] = df[coluna].map(map)
    
    else:
        df[coluna] = df[coluna].astype(str).str.lower()
        
    return pd.get_dummies(df, columns=[coluna],drop_first=True,dtype=int)

# dummies sexo 
x1.sexo.value_counts(dropna=False)

x1 = gerar_dummies(x1,'sexo',{'F':'feminino', 'M':'masculino'})

# estado civil

x1 = gerar_dummies(x1, 'estado_civil', {'Solreiro(a)':'solteiro', 'Casado(a)':'casado','Outro':'outro'})

# raça declarada

x1 = gerar_dummies(x1, 'raca_declarada')

# renda familiar
x1 = gerar_dummies(x1, 'faixa_renda_familiar')

# disciplina
# x1 = gerar_dummies(x1,'id_disciplina')

# curso
x1 = gerar_dummies(x1,'id_curso_cursos')



C:\Users\Alef\AppData\Local\Temp\ipykernel_20136\603150942.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[coluna] = df[coluna].map(map)


In [8]:
x1.situacao.value_counts()

x1 =x1.loc[x1['situacao'].isin(['APROVADO','APROV. C/ DISTINÇÃO','REPROVADO','REP. MEDIA E FALTA', 'REP. FALTA'])]
x1.situacao.value_counts()


x1 = gerar_dummies(x1, 'situacao',{'APROVADO':'aprovado','REPROVADO':'reprovado','REP. MEDIA E FALTA': 'rep_md_flt','REP. FALTA':'rep_flt'})

x1['reprovado'] = (
    x1[['situacao_reprovado',
        'situacao_rep_md_flt',
        'situacao_rep_flt']]
    .sum(axis=1)
    .clip(upper=1)
)

In [9]:

x1['dummy_tecnico'] = (
    ~x1['nome_componete_curricular'].str.contains('º|°', na=False)
).astype(int)


In [10]:
dms = pd.get_dummies(x1['id_disciplina'], drop_first=True, dtype='int8')
print(dms.shape)


print("Shape atual:", x1.shape)
print("N colunas:", len(x1.columns))
print("Disciplinas únicas:", x1['id_disciplina'].nunique())

(357428, 1542)
Shape atual: (357428, 60)
N colunas: 60
Disciplinas únicas: 1543


## Engenharia de Features

## Criando variável de disciplina do ensino técnico

In [11]:
x1['dummy_tecnico'] = (
    ~x1['nome_componete_curricular'].str.contains('º|°', na=False)
).astype(int)

## Reprovações até então e reprovaçãoes até então na disciplina

In [12]:

# criando variável reprovações até então
x1['ano_periodo'] = (x1['ano']+'.'+x1['periodo']).astype(float)

reprov_ate_entao = (
    x1.groupby(['id_discente', 'ano_periodo'])['reprovado']
      .sum()
      .sort_index(level=['id_discente', 'ano_periodo'])
      .groupby(level=0)
      .cumsum()
      .groupby(level=0)
      .shift(1)
      .fillna(0)
      .reset_index(name='reprov_ate_entao')
)

x1 = x1.merge(
    reprov_ate_entao,
    on=['id_discente', 'ano_periodo'],
    how='left'
)
# criando variável reprovações na disciplina até então
reprov_disc_ate_entao = (
    x1.groupby(['id_discente', 'id_disciplina', 'ano_periodo'])['reprovado']
      .sum()
      .sort_index(level=['id_discente', 'id_disciplina', 'ano_periodo'])
      .groupby(level=[0, 1])
      .cumsum()
      .groupby(level=[0, 1])
      .shift(1)
      .fillna(0)
      .reset_index(name='reprov_disc_ate_entao')
)


x1 = x1.merge(
    reprov_disc_ate_entao,
    on=['id_discente', 'id_disciplina', 'ano_periodo'],
    how='left'
)





## Estatística Descritiva

In [13]:

# cópia de segurança
df = x1.copy()

# métricas principais
total_obs = len(df)
total_alunos = df['id_discente'].nunique()
total_disciplinas = df['id_disciplina'].nunique()
taxa_reprovacao = df['reprovado'].mean()

print("Total de observações:", total_obs)
print("Total de alunos:", total_alunos)
print("Total de disciplinas:", total_disciplinas)
print("Taxa de reprovação:", round(taxa_reprovacao*100, 2), "%")

Total de observações: 357428
Total de alunos: 30585
Total de disciplinas: 1543
Taxa de reprovação: 10.79 %


In [100]:
vars_numericas = [
    'reprovado',
    'media_geral',
    'ch_total',
    'ch_integralizada',
    'ch_pendente',
    'reprov_ate_entao',
    'reprov_disc_ate_entao'
]

desc = df[vars_numericas].describe().T
desc['missing_%'] = df[vars_numericas].isnull().mean()*100

print(desc)

                          count         mean          std    min     25%  \
reprovado              357428.0     0.107915     0.310274    0.0    0.00   
media_geral            131992.0     6.781742     2.917343    0.0    5.77   
ch_total               132107.0    48.641344    28.792695    0.0   30.00   
ch_integralizada       120841.0  1529.836968  1213.158846   15.0  600.00   
ch_pendente            120841.0   419.212858   775.872750 -510.0    0.00   
reprov_ate_entao       357428.0     0.272779     1.846030    0.0    0.00   
reprov_disc_ate_entao  357428.0     0.008248     0.099712    0.0    0.00   

                           50%      75%     max  missing_%  
reprovado                 0.00     0.00     1.0   0.000000  
media_geral               8.09     8.75    10.0  63.071724  
ch_total                 40.00    60.00   240.0  63.039549  
ch_integralizada       1305.00  1800.00  4105.0  66.191513  
ch_pendente               0.00   650.00  4030.0  66.191513  
reprov_ate_entao         

In [101]:
vars_comp = [
    'media_geral',
    'ch_total',
    'reprov_ate_entao',
    'reprov_disc_ate_entao',
    'ch_integralizada',
    'ch_pendente'
]

comparacao = df.groupby('reprovado')[vars_comp].mean().T
comparacao.columns = ['Aprovados', 'Reprovados']
comparacao['Diferença'] = comparacao['Reprovados'] - comparacao['Aprovados']

print(comparacao)

                         Aprovados   Reprovados    Diferença
media_geral               8.036073     2.162593    -5.873480
ch_total                 49.654795    44.905282    -4.749513
reprov_ate_entao          0.111975     1.602069     1.490094
reprov_disc_ate_entao     0.004058     0.042881     0.038823
ch_integralizada       1711.584838   422.927067 -1288.657771
ch_pendente             271.064731  1321.488236  1050.423504


In [102]:
sexo = df.groupby('sexo_masculino')['reprovado'].mean().reset_index()
sexo['sexo'] = sexo['sexo_masculino'].map({0:'Feminino',1:'Masculino'})
sexo = sexo[['sexo','reprovado']]

print(sexo)

        sexo  reprovado
0   Feminino   0.083265
1  Masculino   0.235432


In [103]:
tipo_disc = df.groupby('dummy_tecnico')['reprovado'].mean().reset_index()
tipo_disc['tipo'] = tipo_disc['dummy_tecnico'].map({0:'Médio',1:'Técnico'})

print(tipo_disc[['tipo','reprovado']])

      tipo  reprovado
0    Médio   0.172246
1  Técnico   0.103827


In [105]:
curso = (
    df.groupby(['id_curso', 'curso_nome'])['reprovado']
    .agg(['mean','count'])
    .rename(columns={'mean':'taxa_reprov','count':'n'})
)

curso_top10 = curso.sort_values('taxa_reprov', ascending=False).reset_index()

print(curso_top10[['curso_nome','taxa_reprov','n']])

                                           curso_nome  taxa_reprov      n
0                            TÉCNICO EM CITOPATOLOGIA     0.607362    163
1                              TÉCNICO EM SAÚDE BUCAL     0.450413    242
2   TÉCNICO EM REGISTROS E INFORMAÇÕES EM SAÚDE - EaD     0.385816   3779
3   TÉCNICO DE NÍVEL MÉDIO EM PAISAGISMO NA FORMA ...     0.377563   1658
4   TÉCNICO DE NÍVEL MÉDIO EM AGROPECUÁRIA - INTEG...     0.327273   4290
5   TÉCNICO DE NÍVEL MÉDIO EM AQUICULTURA NA FORMA...     0.321859   3980
6   TÉCNICO DE NÍVEL MÉDIO EM LABORATÓRIO DE CIÊNC...     0.307763   1082
7              TÉCNICO EM AGENTE COMUNITÁRIO DE SAÚDE     0.280340    824
8   TÉCNICO DE NÍVEL MÉDIO EM AGROPECUÁRIA NA FORM...     0.231738  11677
9              TÉCNICO EM CUIDADOS DE IDOSOS - PROEJA     0.226744   3612
10                        TÉCNICO EM PRÓTESE DENTÁRIA     0.216678   8658
11                      TÉCNICO EM CUIDADOS DE IDOSOS     0.216157   4543
12  TÉCNICO DE NÍVEL MÉDIO EM VETERINÁ

In [117]:
tempo = (
    df.groupby('ano_periodo')['reprovado']
    .mean()
    .reset_index()
)

print(tempo)

    ano_periodo  reprovado
0        2009.1   0.000000
1        2009.2   0.000000
2        2011.2   0.000000
3        2013.1   0.000000
4        2014.1   1.000000
5        2014.2   1.000000
6        2015.1   0.205025
7        2015.2   0.157876
8        2016.1   0.252444
9        2016.2   0.220133
10       2017.1   0.285129
11       2017.2   0.200949
12       2018.1   0.172294
13       2018.2   0.174312
14       2019.1   0.135308
15       2019.2   0.114580
16       2020.1   0.075173
17       2020.2   0.106246
18       2021.1   0.216971
19       2021.2   0.259731
20       2022.1   0.268480
21       2022.2   0.214669
22       2023.1   0.032945
23       2023.2   0.051280
24       2024.1   0.057877
25       2024.2   0.282997


In [115]:
df.groupby('ano_periodo').size().reset_index(name='n')

,ano_periodo,n
0,2009.1,9
1,2009.2,15
2,2011.2,1
3,2013.1,42
4,2014.1,53
5,2014.2,2
6,2015.1,2746
7,2015.2,5631
8,2016.1,5934
9,2016.2,6169


In [107]:
disc = (
    df.groupby(['id_disciplina', 'nome_componete_curricular'])['reprovado']
    .agg(['mean','count'])
    .rename(columns={'mean':'taxa_reprov','count':'n'})
)

# filtrar disciplinas com pouca observação
disc = disc[disc['n'] >= 30]

disc_rkn = disc.sort_values('taxa_reprov', ascending=False).reset_index()

print(disc_rkn[['nome_componete_curricular','taxa_reprov','n']])

                       nome_componete_curricular  taxa_reprov    n
0                           GEOGRAFIA DA PARAÍBA     0.733333   90
1          PLANEJAMENTO DE ATIVIDADES TURÍSTICAS     0.722222   90
2                             PRIMEIROS SOCORROS     0.722222   90
3                             ÉTICA PROFISSIONAL     0.722222   90
4                  PATRIMÔNIO HISTÓRICO CULTURAL     0.711111   90
..                                           ...          ...  ...
726  NOÇÕES DE CIÊNCIA E TECNOLOGIA DE ALIMENTOS     0.000000   32
727                             GESTÃO AMBIENTAL     0.000000   45
728                        GESTÃO AGROINDUSTRIAL     0.000000   42
729         TRABALHO DE CONCLUSÃO DE CURSO - TCC     0.000000  156
730       ESTÁGIO SUPERVISIONADO EM AGROPECUÁRIA     0.000000  381

[731 rows x 3 columns]


In [49]:
renda_cols = [
    'faixa_renda_familiar_ate_1k',
    'faixa_renda_familiar_2k_4k',
    'faixa_renda_familiar_4k_8k',
    'faixa_renda_familiar_8k_mais'
]

renda_result = {}

for col in renda_cols:
    taxa = df.loc[df[col]==1, 'reprovado'].mean()
    n = df[col].sum()
    renda_result[col] = {'taxa': taxa, 'n': n}

renda_df = pd.DataFrame(renda_result).T
print(renda_df)

                                  taxa        n
faixa_renda_familiar_ate_1k   0.108627  33988.0
faixa_renda_familiar_2k_4k    0.075869   4231.0
faixa_renda_familiar_4k_8k    0.085595    958.0
faixa_renda_familiar_8k_mais  0.151515    363.0


In [118]:
desc.to_csv('desc_variaveis.csv')
comparacao.to_csv('comparacao_aprov_reprov.csv')
curso.to_csv('ranking_cursos.csv')
disc_rkn.to_csv('ranking_disciplinas.csv')
tempo.to_csv('serie_temporal.csv')